## Section E - file.csv

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import TimestampType
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lower, trim, when, regexp_extract

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CERT Analysis") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.executorEnv.SPARK_LOCAL_IP", "127.0.0.1") \
    .getOrCreate()


In [3]:
# utils.py contains all helpers
from helpers import code_quality, parse_date, letters_cleaning

In [4]:
file_raw = spark.read.csv('../data/file.csv', header=True, inferSchema=True)

In [5]:
file_raw.show(3)

+--------------------+-------------------+-------+-------+------------+--------------------+
|                  id|               date|   user|     pc|    filename|             content|
+--------------------+-------------------+-------+-------+------------+--------------------+
|{Q5M2-N0JR22AA-17...|01/02/2010 05:15:35|WCR0044|PC-9174|KQTA0DRL.pdf|25-50-44-46-2D la...|
|{O2W1-I9JA58XQ-79...|01/02/2010 05:16:04|WCR0044|PC-9174|PS5NPZPV.doc|D0-CF-11-E0-A1-B1...|
|{O0T0-H0OD25UT-02...|01/02/2010 05:16:25|WCR0044|PC-9174|VJX6WCC9.doc|D0-CF-11-E0-A1-B1...|
+--------------------+-------------------+-------+-------+------------+--------------------+
only showing top 3 rows


Contains: file operation events (copy, open, delete). <br>
Columns: id, date, user, pc, filename, content <br>
Meaning for GNN: bulk file operations or access to sensitive directories = exfiltration signal

In [6]:
code_quality(file_raw, "file.csv")

Data quality for: file.csv
Number of records: 414556, number of columns: 6
Missing df: [Row(id=0, date=0, user=0, pc=0, filename=0, content=0)]

Duplicates: 0 (0.00%)


{'rows': 414556,
 'columns': 6,
 'duplicates': 0,
 'missing': DataFrame[id: bigint, date: bigint, user: bigint, pc: bigint, filename: bigint, content: bigint]}

In [7]:
# cleaning
file_df = file_raw \
    .dropDuplicates() \
    .filter(F.col("user").isNotNull())

We are now extracting the extension of the file since it can be a threat.

In [8]:
file_df = file_df.withColumn(
    "file_extension",
    F.lower(F.regexp_extract(F.col("filename"), r"\.([a-zA-Z0-9]+)$", 1))
)

In [9]:
file_df.show(3)

+--------------------+-------------------+-------+-------+------------+--------------------+--------------+
|                  id|               date|   user|     pc|    filename|             content|file_extension|
+--------------------+-------------------+-------+-------+------------+--------------------+--------------+
|{B1A4-A4ND13JF-51...|01/02/2010 07:15:58|KBD0201|PC-5997|15ENILPA.doc|D0-CF-11-E0-A1-B1...|           doc|
|{F7M7-Q6ZS16JE-45...|01/03/2010 08:42:38|TGR0912|PC-0817|JKQCU2OE.zip|50-4B-03-04-14 ro...|           zip|
|{R1R0-J3RV24PI-84...|01/03/2010 09:23:50|OSF0896|PC-4020|XZSLDSLA.zip|50-4B-03-04-14 ba...|           zip|
+--------------------+-------------------+-------+-------+------------+--------------------+--------------+
only showing top 3 rows


In [10]:
file_df = parse_date(file_df)

In [11]:
# flag - is the extension a risk factor? 
risky_extensions = ["zip", "rar", "exe", "bat", "sh", "tar", "gz"]
file_df = file_df.withColumn(
    "risky_file",
    F.col("file_extension").isin(risky_extensions)
)

In [12]:
file_df.show(3)

+--------------------+-------------------+-------+-------+------------+--------------------+--------------+-------------------+----+-----+---+----+---------------+-----------------+----------+
|                  id|               date|   user|     pc|    filename|             content|file_extension|          timestamp|year|month|day|hour|day_of_the_week|not_working_hours|risky_file|
+--------------------+-------------------+-------+-------+------------+--------------------+--------------+-------------------+----+-----+---+----+---------------+-----------------+----------+
|{B1A4-A4ND13JF-51...|01/02/2010 07:15:58|KBD0201|PC-5997|15ENILPA.doc|D0-CF-11-E0-A1-B1...|           doc|2010-01-02 07:15:58|2010|    1|  2|   7|              7|             true|     false|
|{F7M7-Q6ZS16JE-45...|01/03/2010 08:42:38|TGR0912|PC-0817|JKQCU2OE.zip|50-4B-03-04-14 ro...|           zip|2010-01-03 08:42:38|2010|    1|  3|   8|              1|            false|      true|
|{R1R0-J3RV24PI-84...|01/03/2010 09

In [29]:
# top 10 extensions
print("Top 10 file extensions:")
file_df.groupBy("file_extension") \
       .count() \
       .orderBy(F.desc("count")) \
       .show(10)

Top 10 file extensions:
+--------------+------+
|file_extension| count|
+--------------+------+
|           doc|265501|
|           pdf| 82437|
|           txt| 21384|
|           jpg| 21313|
|           zip| 21136|
|           exe|  2785|
+--------------+------+



In [30]:
print("Risky extension versus user (top 10):")
file_df.filter(F.col("risky_file")) \
       .groupBy("user") \
       .count() \
       .orderBy(F.desc("count")) \
       .show(10)

Risky extension versus user (top 10):
+-------+-----+
|   user|count|
+-------+-----+
|LGS0292|  636|
|BHV0556|  625|
|OSF0896|  622|
|OMM0111|  550|
|WCR0044|  541|
|BJW0369|  530|
|TDS0246|  516|
|XMB0730|  508|
|JHF0505|  506|
|MHR0057|  504|
+-------+-----+
only showing top 10 rows


In [13]:
file_features = file_df \
    .groupBy("user", "year", "month", "day") \
    .agg(
        F.count("*").alias("amount_of_file_operations"),
        F.sum(F.col("risky_file").cast("int")).alias("risky_files"),
        F.countDistinct("file_extension").alias("different_extensions"),
        F.sum(F.col("not_working_hours").cast("int")).alias("nightly_operations")
    )

In [14]:
file_features.head(3)

[Row(user='ACM0931', year=2010, month=6, day=25, amount_of_file_operations=3, risky_files=0, different_extensions=2, nightly_operations=0),
 Row(user='SBR0157', year=2011, month=5, day=3, amount_of_file_operations=2, risky_files=0, different_extensions=1, nightly_operations=0),
 Row(user='SDH0026', year=2010, month=5, day=25, amount_of_file_operations=23, risky_files=2, different_extensions=4, nightly_operations=0)]

In [17]:
import pandas as pd
import os

def save_chunks(df, name, chunk_size=10000):
    output_dir = f"C:/Users/dagak/test_data/{name}_chunks/"
    os.makedirs(output_dir, exist_ok=True)

    chunks = []
    file_idx = 0

    for i, row in enumerate(df.toLocalIterator()):
        chunks.append(row.asDict())
        
        if (i + 1) % chunk_size == 0:
            pd.DataFrame(chunks).to_parquet(
                f"{output_dir}/{name}_part_{file_idx}.parquet",
                index=False
            )
            print(f"{name}: saved chunk {file_idx} ({i+1} rows)")
            
            chunks = []
            file_idx += 1

    # last 
    if chunks:
        pd.DataFrame(chunks).to_parquet(
            f"{output_dir}/{name}_part_{file_idx}.parquet",
            index=False
        )
        print(f"{name}: saved final chunk {file_idx}")

    print(f"{name} done")

In [18]:
save_chunks(file_features, "file_features")
save_chunks(file_df, "file_clean")

file_features: saved chunk 0 (10000 rows)
file_features: saved chunk 1 (20000 rows)
file_features: saved chunk 2 (30000 rows)
file_features: saved chunk 3 (40000 rows)
file_features: saved final chunk 4
file_features done
file_clean: saved chunk 0 (10000 rows)
file_clean: saved chunk 1 (20000 rows)
file_clean: saved chunk 2 (30000 rows)
file_clean: saved chunk 3 (40000 rows)
file_clean: saved chunk 4 (50000 rows)
file_clean: saved chunk 5 (60000 rows)
file_clean: saved chunk 6 (70000 rows)
file_clean: saved chunk 7 (80000 rows)
file_clean: saved chunk 8 (90000 rows)
file_clean: saved chunk 9 (100000 rows)
file_clean: saved chunk 10 (110000 rows)
file_clean: saved chunk 11 (120000 rows)
file_clean: saved chunk 12 (130000 rows)
file_clean: saved chunk 13 (140000 rows)
file_clean: saved chunk 14 (150000 rows)
file_clean: saved chunk 15 (160000 rows)
file_clean: saved chunk 16 (170000 rows)
file_clean: saved chunk 17 (180000 rows)
file_clean: saved chunk 18 (190000 rows)
file_clean: saved 